# Sign Language Recognition — Phase 1 (Basic CNN, TensorFlow / Keras)

Predicts **ASL hand signs for digits 0-9 and letters a-z** (36 classes) from an image.

**Setup**
- Framework: TensorFlow / Keras
- Dataset: `dataset/asl_dataset/` — 2,515 images, ~70 per class, 400x400 RGB
- Model: small 3-block CNN trained on 64x64 images, kept light so it runs in a few minutes on a local CPU

> **Important — pick the right kernel.** TensorFlow does not have a build for Python 3.14, so it is
> installed in **Python 3.11**. In VS Code, click the kernel picker at the top-right of this notebook and
> choose the **Python 3.11** interpreter, not 3.14. The first cell prints the version so you can confirm.

**How to use:** run every cell top to bottom. The last section lets you pick a photo from your computer
and get a prediction.

## 1. Imports and configuration

In [ ]:
import os, time, random
from pathlib import Path

# Quieten TensorFlow's startup logging (must be set before importing TF)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from keras import layers

from sklearn.model_selection import train_test_split

# --- Configuration knobs (tweak these if you want) ---
DATA_DIR   = Path("dataset/asl_dataset")   # folder with one sub-folder per class
IMG_SIZE   = 64      # images are resized to IMG_SIZE x IMG_SIZE
BATCH_SIZE = 32
EPOCHS     = 20      # enough for this small dataset; raise it if you have time to spare
LR         = 1e-3
VAL_SPLIT  = 0.15    # 70% train / 15% validation / 15% test
TEST_SPLIT = 0.15
SEED       = 42
MODEL_PATH = "sign_cnn_phase1.keras"

# BatchNorm keeps a running average of activation statistics for use at prediction time.
# Keras defaults to 0.99, which needs thousands of steps to settle; this dataset only gives ~55 steps
# per epoch, so the averages would still be near their starting values and predictions would be junk
# even while training accuracy looked fine. 0.9 settles fast enough for a dataset this small.
BN_MOMENTUM = 0.9

# Reproducibility
random.seed(SEED); np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

import sys
print("Python     :", sys.version.split()[0])
print("TensorFlow :", tf.__version__)
print("Keras      :", keras.__version__)
print("GPUs       :", tf.config.list_physical_devices("GPU") or "none (running on CPU)")
print("Dataset folder found:", DATA_DIR.exists())

## 2. Look at the dataset

In [ ]:
class_dirs = sorted([d for d in DATA_DIR.iterdir() if d.is_dir()])
counts = {d.name: len(list(d.glob("*"))) for d in class_dirs}

print(f"Number of classes: {len(class_dirs)}")
print(f"Total images     : {sum(counts.values())}")
print("Images per class :", counts)

plt.figure(figsize=(12, 3))
plt.bar(list(counts.keys()), list(counts.values()), color="#4C78A8")
plt.title("Images per class")
plt.xlabel("class"); plt.ylabel("count")
plt.tight_layout(); plt.show()

In [ ]:
# Show one random sample from a handful of classes
sample_classes = random.sample(class_dirs, 12)

fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for ax, cdir in zip(axes.ravel(), sample_classes):
    img_path = random.choice(list(cdir.glob("*")))
    ax.imshow(Image.open(img_path))
    ax.set_title(f"'{cdir.name}'", fontsize=12)
    ax.axis("off")
plt.suptitle("Random samples from the dataset")
plt.tight_layout(); plt.show()

## 3. Build the data pipeline

The file paths are collected and split **stratified** into train / validation / test, so every class is
represented in all three sets. Each split then becomes a `tf.data` pipeline that reads, decodes and
resizes images on the fly.

Augmentation is not done here — it lives inside the model (next section) as Keras preprocessing layers,
which Keras automatically switches off during evaluation and prediction.

In [ ]:
# Collect every image path and its class index
CLASS_NAMES = [d.name for d in class_dirs]          # ['0','1',...,'9','a','b',...,'z']
NUM_CLASSES = len(CLASS_NAMES)

paths, labels = [], []
for label_idx, cdir in enumerate(class_dirs):
    for img_path in sorted(cdir.glob("*")):
        paths.append(str(img_path))
        labels.append(label_idx)

paths  = np.array(paths)
labels = np.array(labels)

print("Classes:", CLASS_NAMES)
print("Total images:", len(paths))

In [ ]:
# Stratified split: train / val / test
train_p, temp_p, train_y, temp_y = train_test_split(
    paths, labels, test_size=(VAL_SPLIT + TEST_SPLIT), stratify=labels, random_state=SEED)

val_p, test_p, val_y, test_y = train_test_split(
    temp_p, temp_y,
    test_size=TEST_SPLIT / (VAL_SPLIT + TEST_SPLIT),
    stratify=temp_y,
    random_state=SEED)

print(f"Train: {len(train_p)}   Val: {len(val_p)}   Test: {len(test_p)}")


def load_image(path, label):
    """Read a file from disk and turn it into a IMG_SIZE x IMG_SIZE x 3 float tensor (values 0-255)."""
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    return img, label


def make_dataset(p, y, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((p, y))
    if shuffle:
        ds = ds.shuffle(len(p), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


train_ds = make_dataset(train_p, train_y, shuffle=True)
val_ds   = make_dataset(val_p,   val_y)      # kept in a fixed order
test_ds  = make_dataset(test_p,  test_y)     # kept in a fixed order

# Quick check that a batch has the shape we expect
for images, ys in train_ds.take(1):
    print("Batch images:", images.shape, images.dtype, "| range", float(tf.reduce_min(images)),
          "-", float(tf.reduce_max(images)))
    print("Batch labels:", ys.shape, ys.dtype)

## 4. The CNN

A basic architecture. The first two blocks are preprocessing layers that are **only active during
training** (augmentation) or always (rescaling to 0-1):

```
Input 64 x 64 x 3  (values 0-255)
  Augmentation: random rotation / zoom / contrast / brightness   [training only]
  Rescaling 1/255
  Conv2D 32 -> BatchNorm -> ReLU -> MaxPool   ->  32 x 32 x 32   (BN momentum 0.9, see above)
  Conv2D 64 -> BatchNorm -> ReLU -> MaxPool   ->  16 x 16 x 64
  Conv2D 128 -> BatchNorm -> ReLU -> MaxPool  ->   8 x  8 x 128
  Flatten -> Dropout -> Dense 256 -> ReLU -> Dropout -> Dense 36 (softmax)
```

Because rescaling and augmentation live inside the model, predicting on a new photo later needs nothing
more than "load, resize, predict" — no preprocessing to remember or get wrong.

In [ ]:
# Kept deliberately mild. Note there is no random translation: these hands are already centred and
# cropped, and shifting them makes the model noticeably worse rather than better.
data_augmentation = keras.Sequential([
    layers.RandomRotation(0.04),                       # about +/- 14 degrees
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.20),
    layers.RandomBrightness(0.20, value_range=(0, 255)),
], name="augmentation")


def build_model(num_classes: int) -> keras.Model:
    model = keras.Sequential([
        keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),

        data_augmentation,                             # active only while training
        layers.Rescaling(1.0 / 255),

        layers.Conv2D(32, 3, padding="same"),
        layers.BatchNormalization(momentum=BN_MOMENTUM), layers.ReLU(), layers.MaxPooling2D(),

        layers.Conv2D(64, 3, padding="same"),
        layers.BatchNormalization(momentum=BN_MOMENTUM), layers.ReLU(), layers.MaxPooling2D(),

        layers.Conv2D(128, 3, padding="same"),
        layers.BatchNormalization(momentum=BN_MOMENTUM), layers.ReLU(), layers.MaxPooling2D(),

        layers.Flatten(),
        layers.Dropout(0.3),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax"),
    ], name="sign_cnn")
    return model


model = build_model(NUM_CLASSES)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR),
    loss="sparse_categorical_crossentropy",            # labels are integers, not one-hot
    metrics=["accuracy"],
)

model.summary()

## 5. Train

In [ ]:
callbacks = [
    # Keep only the weights that scored best on the validation set
    keras.callbacks.ModelCheckpoint(
        MODEL_PATH, monitor="val_accuracy", save_best_only=True, verbose=1),

    # Halve the learning rate when validation accuracy stops improving
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_accuracy", factor=0.5, patience=3, verbose=1),
]

start = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

print(f"\nTraining finished in {time.time() - start:.1f}s")
print(f"Best validation accuracy: {max(history.history['val_accuracy']):.3f}")
print(f"Best model saved to: {Path(MODEL_PATH).resolve()}")

## 6. Training curves

In [ ]:
h = history.history
epochs_range = range(1, len(h["loss"]) + 1)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(epochs_range, h["loss"],     label="train")
ax[0].plot(epochs_range, h["val_loss"], label="validation")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)

ax[1].plot(epochs_range, h["accuracy"],     label="train")
ax[1].plot(epochs_range, h["val_accuracy"], label="validation")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)

plt.tight_layout(); plt.show()

## 7. Evaluate on the held-out test set

Note the training accuracy above will look *lower* than the validation accuracy. That is expected here —
augmentation is applied during training but switched off during evaluation, so the model is being scored
on harder images than it is graded on.

In [ ]:
# Reload the best checkpoint before testing
model = keras.models.load_model(MODEL_PATH)

test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Test loss: {test_loss:.4f}   Test accuracy: {test_acc:.3f}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# test_ds is not shuffled, so predictions line up with test_y
probs  = model.predict(test_ds, verbose=0)
y_pred = probs.argmax(axis=1)
y_true = test_y

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 9))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion matrix (test set)")
plt.xlabel("predicted"); plt.ylabel("true")
plt.xticks(range(NUM_CLASSES), CLASS_NAMES, fontsize=7)
plt.yticks(range(NUM_CLASSES), CLASS_NAMES, fontsize=7)
plt.colorbar(); plt.tight_layout(); plt.show()

In [ ]:
# Visual sanity check on one batch of test images
images, ys = next(iter(test_ds))
preds = model.predict(images, verbose=0).argmax(axis=1)

fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for ax, img, t, p in zip(axes.ravel(), images, ys.numpy(), preds):
    ax.imshow(img.numpy().astype("uint8"))
    ax.set_title(f"true {CLASS_NAMES[t]} / pred {CLASS_NAMES[p]}",
                 fontsize=9, color="green" if t == p else "red")
    ax.axis("off")
plt.suptitle("Test predictions (green = correct, red = wrong)")
plt.tight_layout(); plt.show()

## 8. Predict on your own photo

Run the next cell to define the prediction helper, then run the cell after it to **pick an image file
from your computer**. A file-picker window opens — it may appear behind VS Code, so check your taskbar.

If the picker does not work on your machine, use the manual cell at the bottom instead: just set
`MY_IMAGE` to the path of your photo.

In [ ]:
def predict_image(image_path, top_k=3, show=True):
    """Load an image from disk, run it through the trained CNN, show the top-k predictions."""
    image_path = Path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(f"No such file: {image_path}")

    original = Image.open(image_path).convert("RGB")

    # Same preparation the pipeline uses: resize to IMG_SIZE, keep values 0-255.
    # Rescaling happens inside the model, so nothing else is needed here.
    img = original.resize((IMG_SIZE, IMG_SIZE))
    x = np.expand_dims(np.array(img, dtype="float32"), axis=0)   # shape: 1 x 64 x 64 x 3

    probs = model.predict(x, verbose=0)[0]

    top_i   = probs.argsort()[::-1][:top_k]
    results = [(CLASS_NAMES[i], float(probs[i])) for i in top_i]

    if show:
        fig, ax = plt.subplots(1, 2, figsize=(11, 4))

        ax[0].imshow(original); ax[0].axis("off")
        ax[0].set_title(f"Prediction: '{results[0][0]}'  ({results[0][1]*100:.1f}% confident)",
                        fontsize=13)

        names = [r[0] for r in results][::-1]
        vals  = [r[1] for r in results][::-1]
        ax[1].barh(names, vals, color="#4C78A8")
        ax[1].set_xlim(0, 1); ax[1].set_xlabel("probability")
        ax[1].set_title(f"Top {top_k} predictions")
        for i, v in enumerate(vals):
            ax[1].text(min(v + 0.02, 0.88), i, f"{v*100:.1f}%", va="center")

        plt.tight_layout(); plt.show()

    print("Top predictions:")
    for name, p in results:
        print(f"  '{name}' : {p*100:5.1f}%")
    return results

In [ ]:
# ---- Upload / choose a photo -------------------------------------------------
# Opens a file dialog. Select an image of a hand sign and the model predicts it.
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)          # try to bring the dialog to the front

MY_IMAGE = filedialog.askopenfilename(
    title="Select a sign-language image",
    filetypes=[("Image files", "*.jpg *.jpeg *.png *.bmp *.webp"), ("All files", "*.*")],
)
root.destroy()

if MY_IMAGE:
    print("Selected:", MY_IMAGE)
    predict_image(MY_IMAGE)
else:
    print("No file selected. Re-run this cell, or use the manual cell below.")

### Manual alternative (no file dialog)

Paste the full path to your image below and run the cell. On Windows keep the `r` before the quotes,
for example `r"C:\Users\arpan\Pictures\my_sign.jpg"`.

In [ ]:
MY_IMAGE = r"dataset/asl_dataset/b/hand1_b_bot_seg_1_cropped.jpeg"   # <-- change this path

predict_image(MY_IMAGE)

---

## Notes and what comes next

Some honest caveats about this Phase 1 model:

- The dataset is small (~70 images per class) and every image is a **tightly cropped hand on a plain
  background**. The model is most accurate on photos that look like that. A wide shot with a busy
  background, or a hand at an unusual angle, will confuse it — cropping your photo close to the hand
  before predicting makes a big difference.
- ASL `j` and `z` involve motion, which a still image cannot fully capture.
- Visually similar signs — `m`/`n`/`s`, `a`/`e`, `1`/`d`, `2`/`v` — are the usual sources of error. The
  confusion matrix above shows which ones this particular run struggled with.

**Ideas for Phase 2:** transfer learning (`keras.applications.MobileNetV2` / `ResNet50`), hand detection
and auto-cropping (MediaPipe) so raw photos get framed like the training data, stronger augmentation,
more data, and live webcam inference.